# Backpropagation Deep Dive

**What you'll build:** Individual ops with forward + backward, numerical gradient verification.

**Prerequisites:** [neural-net-from-scratch.ipynb](neural-net-from-scratch.ipynb), calculus chain rule.

## Concept Recap

**Chain rule:** $\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial x}$

**Computation graph:** each op stores its inputs (for backward). Gradient flows backward from loss.

**Key op gradients:**
- Add: $\frac{\partial}{\partial x}(x + y) = 1$ — gradient passes through unchanged
- Multiply: $\frac{\partial}{\partial x}(xy) = y$ — other operand
- ReLU: $\frac{\partial}{\partial x}\max(0,x) = \mathbf{1}[x > 0]$
- MatMul $Y = XW$: $\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y} W^T$, $\frac{\partial L}{\partial W} = X^T \frac{\partial L}{\partial Y}$

**Numerical gradient check:** verify analytical gradient via finite differences:
$$\frac{\partial f}{\partial x_i} \approx \frac{f(x+\epsilon e_i) - f(x-\epsilon e_i)}{2\epsilon}$$
Analytical ≈ numerical within relative error 1e-5 means backprop is correct.

In [ ]:
import numpy as np

class ReLU:
    def forward(self, x): self.x = x; return np.maximum(0, x)
    def backward(self, dout): return dout * (self.x > 0)

class MatMul:
    def forward(self, X, W): self.X, self.W = X, W; return X @ W
    def backward(self, dout): return dout @ self.W.T, self.X.T @ dout

class Add:
    def forward(self, x, b): return x + b
    def backward(self, dout): return dout, dout.sum(0)  # broadcast backward

def numerical_grad(f, x, eps=1e-5):
    grad = np.zeros_like(x)
    for i in range(x.size):
        x_plus = x.copy(); x_plus.flat[i] += eps
        x_minus = x.copy(); x_minus.flat[i] -= eps
        grad.flat[i] = (f(x_plus) - f(x_minus)) / (2 * eps)
    return grad

# Gradient check on a simple 2-layer forward pass
np.random.seed(0)
X = np.random.randn(4, 3)
W1 = np.random.randn(3, 5)
b1 = np.random.randn(5)
W2 = np.random.randn(5, 2)
b2 = np.random.randn(2)

mm1, add1, relu, mm2, add2 = MatMul(), Add(), ReLU(), MatMul(), Add()

def forward(W1_):
    h1 = mm1.forward(X, W1_)
    h2 = add1.forward(h1, b1)
    h3 = relu.forward(h2)
    h4 = mm2.forward(h3, W2)
    out = add2.forward(h4, b2)
    return np.sum(out**2)  # dummy scalar loss

analytical_W1 = np.zeros_like(W1)
dout = 2 * add2.forward(mm2.forward(relu.forward(add1.forward(mm1.forward(X, W1), b1)), W2), b2)
db2 = dout.sum(0)
dh4, _ = add2.backward(dout)
dh3, dW2_a = mm2.backward(dh4)
dh2 = relu.backward(dh3)
dh1, db1_a = add1.backward(dh2)
_, dW1_a = mm1.backward(dh1)

num_W1 = numerical_grad(forward, W1)
rel_error = np.abs(dW1_a - num_W1) / (np.abs(dW1_a) + np.abs(num_W1) + 1e-9)
print(f"Max relative error (W1): {rel_error.max():.2e}  (should be < 1e-5)")

In [ ]:
# Train a 3-layer MLP and verify convergence
from sklearn.datasets import make_moons

X_data, y_data = make_moons(n_samples=300, noise=0.1, random_state=42)

np.random.seed(42)
W1_ = np.random.randn(2, 16) * 0.1
b1_ = np.zeros(16)
W2_ = np.random.randn(16, 2) * 0.1
b2_ = np.zeros(2)
lr = 0.05
losses = []

def softmax(x): e = np.exp(x - x.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)

for _ in range(1000):
    h = np.maximum(0, X_data @ W1_ + b1_)
    out = softmax(h @ W2_ + b2_)
    loss = -np.log(out[np.arange(len(y_data)), y_data] + 1e-9).mean()
    losses.append(loss)
    dout = out; dout[np.arange(len(y_data)), y_data] -= 1; dout /= len(y_data)
    dW2_ = h.T @ dout; db2_ = dout.sum(0)
    dh = dout @ W2_.T * (h > 0)
    dW1_ = X_data.T @ dh; db1_ = dh.sum(0)
    W1_ -= lr * dW1_; b1_ -= lr * db1_
    W2_ -= lr * dW2_; b2_ -= lr * db2_

preds = softmax(np.maximum(0, X_data @ W1_ + b1_) @ W2_ + b2_).argmax(1)
print(f"Train accuracy: {np.mean(preds == y_data):.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(losses)
plt.xlabel('Iteration'); plt.ylabel('Loss')
plt.title('MLP Training Loss (Moons dataset)')
plt.tight_layout(); plt.show()

## Exercises

1. **Sigmoid backward:** Derive $\frac{d\sigma}{dx} = \sigma(x)(1-\sigma(x))$. Implement a Sigmoid op class and gradient-check it.
2. **Cross-entropy backward:** Derive the gradient of $-\sum y_k \log \hat{p}_k$ w.r.t. logits. Implement and verify.
3. **Batch norm backward:** Implement the BatchNorm backward pass (see the original batch norm paper for the full derivation).

## Summary

- Backprop = chain rule applied to a computation graph
- Numerical gradient check is the canonical way to verify an analytical gradient implementation
- Relative error < 1e-5 confirms correctness; larger errors indicate a bug in backward()

**Next:** [CNN Image Classifier](cnn-image-classifier.ipynb)